# Шаг 8. Прогноз

Прогноз выручки на 30 дней вперёд и честная проверка его точности. Функции из `app/forecast.py`.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

from app.config import SYNTHETIC_DIR
from app.forecast import (
    choose_horizon,
    compare_models,
    evaluate,
    horizon_scan,
    naive_forecast,
    prophet_forecast,
    to_daily,
)
from app.forecast_report import quality_report
from app.pipeline import prepare_events


def read_mapping(file_name: str) -> dict[str, str]:
    # Эталон разметки колонок из .json рядом с тестовым файлом
    return json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))


shop, _ = prepare_events(SYNTHETIC_DIR / "shop.xlsx", read_mapping("shop.xlsx"))
cafe, _ = prepare_events(SYNTHETIC_DIR / "cafe.csv", read_mapping("cafe.csv"))
salon, _ = prepare_events(SYNTHETIC_DIR / "salon.xlsx", read_mapping("salon.xlsx"))

## 1. to_daily: ряд по дням для прогноза

In [ ]:
# Две колонки: ds - дата, y - выручка за день. Так их требует Prophet
shop_daily = to_daily(shop)
print("Дней:", len(shop_daily))
shop_daily.head()

In [ ]:
# У салона визиты со временем (09:00, 18:00...).
# В ряду по дням время уходит: все визиты одного дня складываются
salon_daily = to_daily(salon)
print("Визиты в салоне:", len(salon), "| дней в ряду:", len(salon_daily))
salon_daily.tail(7)

## 2. naive_forecast: простой прогноз "как на прошлой неделе"

In [ ]:
# Последние 7 дней повторяются по кругу 30 дней вперёд.
# Сравни: первые 7 строк прогноза = последние 7 дней ряда выше
salon_naive = naive_forecast(salon_daily)
salon_naive.head(10)

In [ ]:
# Слабость простого прогноза на графике: случайный всплеск и провал последней недели
# (1 140 в пятницу и 90 во вторник) повторяются КАЖДУЮ неделю. Поэтому он - только точка отсчёта
import matplotlib.pyplot as plt

history = salon_daily.tail(60)
plt.figure(figsize=(10, 4))
plt.plot(history["ds"], history["y"], label="факт")
plt.plot(salon_naive["ds"], salon_naive["yhat"], label="простой прогноз", linestyle="--")
plt.legend()
plt.title("Салон: последние 60 дней и простой прогноз на 30")
plt.grid(alpha=0.3);

## 3. prophet_forecast: прогноз Prophet

Продажи = тренд + сезонность (неделя, год) + праздники Беларуси + шум. Prophet находит эти части в истории и продолжает их в будущее.

In [ ]:
# Прогноз магазина на 30 дней: yhat - прогноз, yhat_lower / yhat_upper - коридор 80%.
# Пятница-воскресенье выше остальных дней: недельную сезонность Prophet нашёл сам
shop_prophet = prophet_forecast(shop_daily)
print(f"Итого за 30 дней: {shop_prophet['yhat'].sum():,.0f}")
print(f"Коридор: {shop_prophet['yhat_lower'].sum():,.0f} - {shop_prophet['yhat_upper'].sum():,.0f}")
shop_prophet.head(7).round({"yhat": 1, "yhat_lower": 1, "yhat_upper": 1})

In [ ]:
# Факт за 60 дней, прогноз Prophet и коридор (красивый график сделаем в forecast_chart)
import matplotlib.pyplot as plt

history = shop_daily.tail(60)
plt.figure(figsize=(10, 4))
plt.plot(history["ds"], history["y"], label="факт")
plt.plot(shop_prophet["ds"], shop_prophet["yhat"], label="прогноз Prophet")
plt.fill_between(
    shop_prophet["ds"],
    shop_prophet["yhat_lower"],
    shop_prophet["yhat_upper"],
    alpha=0.2,
    label="коридор 80%",
)
plt.legend()
plt.title("Магазин: прогноз Prophet на 30 дней")
plt.grid(alpha=0.3);

In [ ]:
# Салон: данных год и они шумные - коридор намного шире, чем у магазина.
# Это честно: чем меньше и неровнее история, тем меньше уверенность
salon_prophet = prophet_forecast(salon_daily)
print(f"Салон, итого за 30 дней: {salon_prophet['yhat'].sum():,.0f}")
print(
    f"Коридор: {salon_prophet['yhat_lower'].sum():,.0f} - {salon_prophet['yhat_upper'].sum():,.0f}"
)
print(f"Простой прогноз: {salon_naive['yhat'].sum():,.0f}")

## 4. evaluate и compare_models: экзамен на прошлом

Прячем конец истории, прогнозируем его по тому, что было раньше, и сравниваем с правдой. Несколько раз на разных отрезках.

- **MAE** - средняя ошибка за день, в рублях
- **WAPE_%** - ошибка в % от продаж
- **ошибка_итога_%** - насколько ошибся в сумме за весь период (+ завысил, - занизил)
- **в_коридоре_%** - сколько реальных дней попало в коридор (обещано 80%)

In [ ]:
# Экзамен для Prophet на магазине: 3 раза прячем по 30 дней, прогнозируем, сравниваем, усредняем.
# Функцию прогноза передаём как параметр - так один экзамен проверит любую модель
evaluate(shop_daily, prophet_forecast)

In [ ]:
# Сравнение моделей одним экзаменом: лучшая (меньше WAPE) - сверху.
# Нюанс магазина: по дням Prophet точнее (8% против 10,6%), а по ИТОГУ за месяц
# простой прогноз ошибся меньше (0,2% против 4,2%) - ошибки по дням у него взаимно погасились
compare_models(shop_daily)

In [ ]:
# Главная таблица для курсовой: все три бизнеса, обе модели, один экзамен
for name, events in [("магазин", shop), ("кафе", cafe), ("салон", salon)]:
    print(name)
    print(compare_models(to_daily(events)).to_string())
    print()

## 5. quality_report: понятный вывод для пользователя

Можно ли верить прогнозу и почему. Причины не придумываются, а вычисляются из данных и результатов экзамена.

In [ ]:
# Что увидит владелец каждого бизнеса: точность словами и причины - с цифрами из его данных
for name, events in [("магазин", shop), ("кафе", cafe), ("салон", salon)]:
    daily = to_daily(events)
    report = quality_report(daily, compare_models(daily))
    print(f"=== {name}: точность {report['точность']} ===")
    print(report["вывод"])
    for reason in report["причины"]:
        print("  -", reason)
    print()

In [ ]:
# Короткая история: обрежем магазин до 120 дней - появится честная причина
short_shop = shop_daily.tail(120)
short_report = quality_report(short_shop, compare_models(short_shop))
print(short_report["вывод"])
for reason in short_report["причины"]:
    print("  -", reason)

## 6. horizon_scan и choose_horizon: на сколько дней вперёд можно верить

Экзамен на 7, 14, 30, 60, 90 днях. Выбираем самый дальний горизонт, где ошибка по дням до 35% **и** ошибка итога до 15%. Горизонт не дальше трети истории.

In [ ]:
# Магазин: экзамен на 7, 14, 30, 60, 90 днях.
# ЗАМЕТЬ: ошибка почти не растёт с горизонтом - тестовые данные слишком "правильные"
# (идеально ровная сезонность). На реальных данных ошибка с горизонтом растёт заметно
shop_scan = horizon_scan(shop_daily)
print("Выбран горизонт:", choose_horizon(shop_scan), "дней")
shop_scan

In [ ]:
# Салон: по дням ошибка держится около 33% (ниже порога 35%), но ИТОГ уезжает:
# 10% на 30 днях, почти 18% на 60. Поэтому горизонт - 30 дней, а не 90
salon_scan = horizon_scan(salon_daily)
print("Выбран горизонт:", choose_horizon(salon_scan), "дней")
salon_scan

In [ ]:
# Короткая история: горизонт не дальше трети истории.
# 120 дней -> не дальше 40 (из списка - 30), 60 дней -> не дальше 20 (из списка - 14)
for days in [120, 60]:
    short_scan = horizon_scan(shop_daily.tail(days))
    print(f"История {days} дн. -> горизонт {choose_horizon(short_scan)} дн.")

### Попробуй сломать

In [ ]:
# Меньше недели истории - повторять нечего
naive_forecast(salon_daily.head(5))

In [ ]:
# Для экзамена нужно хотя бы 30 дней на обучение + проверяемый отрезок
evaluate(shop_daily.head(40), prophet_forecast)